# Rule: **solve_sector_network**

**Outputs**

- results/networks/`base_s_{clusters}_{opts}_{sector_opts}_{horizon}.nc`

In [ ]:
######################################## Parameters

### Run
name = ''
prefix = ''

### Network
clusters = 5
opts = ''
sector_opts = ''
horizon = '2030'

### Spatial domain 'ES' or 'EU' (for maps domain and NUTS regions)
spatial_domain = 'ES'

In [ ]:
##### Import packages
import pypsa
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os 
import sys
import numpy as np


##### Import local functions
sys.path.append(os.path.abspath(os.path.join("..")))
import functions as xp


##### Read params.yaml
params = xp.read_params("../params.yaml")


##### Ignore warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


##### Region files
region_tag = f"base_s_{clusters}"
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)


##### NUTS files (used here to display results at NUTS level)
gdf_NUTS2, gdf_NUTS3 = xp.load_nuts(params, spatial_domain=spatial_domain)

## `base_s_{clusters}_{opts}_{sector_opts}_{horizon}.nc`

Load the network and show its components.

In [ ]:
file = f"base_s_{clusters}_{opts}_{sector_opts}_{horizon}.nc"
n = xp.load_network(
    params,
    file,
    prefix=prefix,
    name=name,
    location="results",
)

n

Plot the initial and optimal networks.

In [ ]:
#################### Parameters
line_widths_0 = 1*n.lines.s_nom / 1e3
link_widths_0 = 1*n.links.p_nom / 1e3

line_widths_1 = 1*n.lines.s_nom_opt / 1e3
link_widths_1 = 1*n.links.p_nom_opt / 1e3



#################### Figure
fig_size = [12,12]
crs = ccrs.PlateCarree()

fig, axes = plt.subplots(1, 2, figsize=fig_size, subplot_kw={'projection': crs})


##### Initial network
ax0 = axes[0]

### Add network
n.plot(ax=ax0, 
       line_widths=line_widths_0, 
       link_widths=link_widths_0, 
       bus_sizes=params['bus_sizes'], 
       bus_colors=params['bus_colors'], 
       boundaries=params[f'boundaries_offshore_{spatial_domain}'])

### Add regions_onshore
xp.map_add_region(ax0, gdf_regions_onshore, params['map_add_region'])

### Add regions_offshore
xp.map_add_region(ax0, gdf_regions_offshore, params['map_add_region'], is_offshore=True)

### Add map features
xp.map_add_features(ax0, params['map_add_features'])



##### Optimal network
ax1 = axes[1]

### Add network
n.plot(ax=ax1, 
       line_widths=line_widths_1, 
       link_widths=link_widths_1, 
       bus_sizes=params['bus_sizes'], 
       bus_colors=params['bus_colors'], 
       boundaries=params[f'boundaries_offshore_{spatial_domain}'])

### Add regions_onshore
xp.map_add_region(ax1, gdf_regions_onshore, params['map_add_region'])

### Add regions_offshore
xp.map_add_region(ax1, gdf_regions_offshore, params['map_add_region'], is_offshore=True)

### Add map features
xp.map_add_features(ax1, params['map_add_features'])

### Variable: `n.buses`

Place `n.buses` in a dataFrame and show its content.

In [ ]:
bb = n.buses

bb.head()

### Variable: `n.carriers`

Place `n.carriers` in a dataFrame and show its content.

In [ ]:
cc = n.carriers

cc.head()

### Variable: `n.generators`

Place `n.generators` in a dataFrame and show its content.

In [ ]:
gg = n.generators

gg.head()

#### Summary

What is the aggregated capacity, initial and optimal, and potential per carrier? 

In [ ]:
gg.groupby('carrier').agg(
    Total_capacity=pd.NamedAgg(column='p_nom', aggfunc='sum'),
    Optimal_capacity=pd.NamedAgg(column='p_nom_opt', aggfunc='sum'),
    Total_max_capacity=pd.NamedAgg(column='p_nom_max', aggfunc='sum'),
)

#### Summary by resource class

Solar and wind carries may have classes. Show a table with a feature per bus and class.

In [ ]:
#################### Parameters

### Select carrier
carrier = 'onwind'

### Select feature (uncomment one of the following):
#feature = 'p_nom'
#feature = 'p_nom_max'
feature = 'p_nom_opt'



#################### Table
if ('wind' in carrier or 'solar' in carrier):   
    ### Filter
    gg_filtered = gg[gg['carrier']==carrier][['bus', 'p_nom', 'p_nom_max', 'p_nom_opt']]
    ### Add class column
    gg_filtered['resource_class'] = gg_filtered.index.to_series().str.split(" ").str[-2]
    ### MW to GW
    gg_filtered[['p_nom', 'p_nom_max', 'p_nom_opt']] = gg_filtered[['p_nom', 'p_nom_max', 'p_nom_opt']]/1000
    ### Make pivot
    gg_pivot = gg_filtered.pivot(index="bus", columns="resource_class", values=feature)

    styled_table = gg_pivot.style \
        .set_caption(f"Table with {feature} [GW] for {carrier}, according to bus and class.") \
        .format("{:.2f}") \
        .background_gradient(cmap="Blues")  
    
    display(styled_table)

else:
     print(f'Carrier {carrier} is not solar or wind type.. does it contain classes?')



Add a bar plot showing `p_nom`, `p_nom_opt` and `p_nom_max`

In [ ]:
#################### Parameters

## bar width
width = 0.3 




#################### Figure

df_sorted = gg_filtered.sort_values(["bus", "resource_class"])

buses = df_sorted["bus"].unique()
resources = df_sorted["resource_class"].unique()

x = np.arange(len(buses))                 
offsets = np.linspace(-width/2.5, width/2.5, len(resources))  


fig, ax = plt.subplots(figsize=(12, 8))

colors = {
    "p_nom_max": "#cfe2f3",
    "p_nom_opt": "#4daf4a",
    "p_nom": "#08306b"
}


metrics_order = ["p_nom_max", "p_nom_opt", "p_nom"]

# To control added labels
labels_added = set()

for i, res in enumerate(resources):
    df_res = df_sorted[df_sorted["resource_class"] == res]
    pos = x + offsets[i]
    
    for metric in metrics_order:
        label = metric if metric not in labels_added else None
        ax.bar(
            pos,
            df_res[metric],
            width=width/len(resources),
            color=colors[metric],
            alpha=0.7,
            label=label
        )
        labels_added.add(metric)

ax.set_xticks(x)
ax.grid(axis="y", linestyle="--", alpha=0.6)
ax.set_xlabel("Bus", fontsize=16)
ax.set_ylabel("Capacity [GW]", fontsize=16)
ax.set_xticklabels(buses, fontsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.legend(fontsize=14)

plt.tight_layout()


#### Maps

Plot a map showing a particular feature of a generation carrier at each region.

Then, show another map with the same information aggregated to a certain NUTS level (weighted with ovelap area).

In [ ]:
#################### Parameters

### Select carrier
carrier = 'onwind'

### For solar & onwind carriers, select the class number ('all' to aggregate all classes)
resource_class = 'all'

### Select feature (uncomment one of the following):
# feature = 'area' 
# feature = 'p_nom'
# feature = 'p_nom_density'
# feature = 'p_nom_max'
# feature = 'p_nom_max_density'
#feature = 'p_nom_max_ratio'
feature = 'p_nom_opt'
# feature = 'p_nom_opt_density'
# feature = 'p_nom_opt_max_ratio'



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Define gdf_regions
if 'off' in carrier:
    gdf_regions = gdf_regions_offshore
else:
    gdf_regions = gdf_regions_onshore


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_generators(carrier, n, feature, ax, gdf_regions, resource_class, params['map_network_generators'], params_local)

In [ ]:
#################### Parameters

### Select NUTS level
NUTS_level = 2



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



if 'off' in carrier:
    print('Aggregation at NUTS level for offshore is not possible')
else:
    #################### Figure
    fig_size = [12,6]
    crs = ccrs.PlateCarree()

    fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


    ### Define gdf_regions
    gdf_regions = gdf_regions_onshore


    ### Define NUTS file
    if NUTS_level==2:
        gdf_NUTS = gdf_NUTS2
    if NUTS_level==3:
        gdf_NUTS = gdf_NUTS3    


    ### Add map features
    xp.map_add_features(ax, params['map_add_features'])

    ### Add network feature aggregated at NUTS regions
    xp.map_NUTS_generators(carrier, n, f'{feature}_NUTS', ax, gdf_regions, gdf_NUTS, resource_class, params['map_NUTS_generators'], params_local)


#### Costs

What is the capital cost of generators?

In [ ]:
#################### Parameters

### Select carriers
carrier_list = ['onwind', 'solar', 'offwind-float']



#################### Figure
fig_size = [10,5]
fig, ax = plt.subplots(1,1,figsize=fig_size)

# Give a message with the carriers not considered
carrier_all = gg['carrier'].unique().tolist()
carrier_excluded = [carrier for carrier in carrier_all if carrier not in carrier_list]
print(f'Carriers ommitted: {carrier_excluded}')

# Define bins
minimo = 0 # 0.95 * gg.loc[ gg['carrier'].isin(carrier_list), 'capital_cost'].min()
maximo = 1.05 * gg.loc[ gg['carrier'].isin(carrier_list), 'capital_cost'].max()
ancho = 1000

bins = np.arange(minimo, maximo, 1000)

# Define colors
tech_colors = n.carriers['color']


for carrier in carrier_list:

    ##### Filter the carrier
    df = gg[gg['carrier']==carrier]


    ##### Only single cost for the carrier
    if df['capital_cost'].round(2).nunique()==1:

        valor = df['capital_cost'].unique()[0]

        ax.axvline(x=valor, label=carrier, color = tech_colors[carrier])

        print(f'Capital cost for {carrier} is: {valor:.2f} EUR/MW·year')

    ##### Different costs for the carrier
    else:
        plt.hist(df['capital_cost'], bins=bins, 
                 edgecolor='none', color = tech_colors[carrier],
                 label=carrier, alpha=1)
        
        valor = df['capital_cost'].mean()
        print(f'Average capital cost for {carrier} is: {valor:.2f} EUR/MW·year')


ax.set_title('capital cost')
ax.set_xlabel('EUR/(MW·year)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

### Variable: `n.generators_t[p_max_pu]`

Place `n.generators_t[p_max_pu]` in a dataFrame and show its content.

In [ ]:
ggt_pmaxpu = n.generators_t['p_max_pu']

ggt_pmaxpu.head()

#### Time series

How do the potential generation time series look like?

In [ ]:
#################### Parameters

### Select carrier
carrier = 'onwind'

### For solar & onwind carriers, select the class number ('all' to plot all classes individually)
resource_class = 'all'
    
### Define period
start = '2023-03-01'
end = '2023-03-10'



#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)

if ('wind' in carrier or 'solar' in carrier) and resource_class != 'all':
    ggt_pmaxpu.loc[start:end].filter(like=f'{resource_class} {carrier}').plot(ax=ax, alpha=.7, legend=False, linewidth=.5)
else:
    ggt_pmaxpu.loc[start:end].filter(like=carrier).plot(ax=ax, alpha=.7, legend=False, linewidth=.5)

ax.grid(True, linestyle='--', alpha=0.25)
ax.set_ylabel('')

#### Summary by resource class

Solar and wind carries may have classes. Show a table with a feature per bus and class.

In [ ]:
#################### Parameters

### Select carrier
carrier = 'onwind'

### Select feature (uncomment one of the following):
feature = 'CF'



#################### Table
if ('wind' in carrier or 'solar' in carrier):   
    ### Filter
    ggt_pmaxpu_filtered = ggt_pmaxpu.filter(like=carrier, axis=1).mean().to_frame(name='CF')
    ### Add class and bus columns. Use multi-index with 'bus' and 'carrier'    
    split_index = ggt_pmaxpu_filtered.index.to_series().str.rsplit(' ', n=2, expand=True)
    ggt_pmaxpu_filtered['bus'] = split_index[0].values
    ggt_pmaxpu_filtered['resource_class'] = split_index[1].astype(int) # resource_class as integer    
    ### Make pivot
    ggt_pmaxpu_pivot = ggt_pmaxpu_filtered.pivot(index="bus", columns="resource_class", values=feature)

    styled_table = ggt_pmaxpu_pivot.style \
        .set_caption(f"Table with {feature} for {carrier}, according to bus and class.") \
        .format("{:.3f}") \
        .background_gradient(cmap="Blues")
    
    display(styled_table)
else:
    print(f'Carrier {carrier} is not solar or wind type.. does it contain pmax_pu and classes?')


Add a bar plot showing `CF`

In [ ]:
#################### Parameters

## bar width
width = 0.3 



#################### Figure

df_sorted = ggt_pmaxpu_filtered.sort_values(["bus", "resource_class"])

buses = df_sorted["bus"].unique()
resources = df_sorted["resource_class"].unique()

x = np.arange(len(buses))                 
offsets = np.linspace(-width/2.5, width/2.5, len(resources))  


fig, ax = plt.subplots(figsize=(12, 8))

colors = {
    "CF": "#bc91d8",
}


metrics_order = ["CF"]

# To control added labels
labels_added = set()

for i, res in enumerate(resources):
    df_res = df_sorted[df_sorted["resource_class"] == res]
    pos = x + offsets[i]
    
    for metric in metrics_order:
        label = metric if metric not in labels_added else None
        ax.bar(
            pos,
            df_res[metric],
            width=width/len(resources),
            color=colors[metric],
            alpha=0.7,
            label=label
        )
        labels_added.add(metric)

ax.set_xticks(x)
ax.grid(axis="y", linestyle="--", alpha=0.6)
ax.set_xlabel("Bus", fontsize=16)
ax.set_ylabel("CF", fontsize=16)
ax.set_xticklabels(buses, fontsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.legend(fontsize=14)

plt.tight_layout()


#### Maps

Plot a map showing a particular feature of the potential generation for a carrier at each region.

In [ ]:
#################### Parameters

##### Select carrier:
carrier = 'onwind'

### For solar & onwind carriers, select the class number (note that here, 'all' is not valid)
resource_class = 0

##### Select feature (uncomment one of the following):
feature = 'CF' 



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Define gdf_regions
if 'off' in carrier:
    gdf_regions = gdf_regions_offshore
else:
    gdf_regions = gdf_regions_onshore


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_generatorst_pmaxpu(carrier, n, feature, ax, gdf_regions, params['map_network_generatorst_pmaxpu'], params_local, resource_class)

### Variable: `n.generators_t[p]`

Place `n.generators_t[p]` in a dataFrame and show its content.

In [ ]:
ggt_p = n.generators_t['p']

ggt_p.head()

#### Time series

How do the dispatched generation time series look like?

In [ ]:
#################### Parameters
carrier = 'onwind'

### For solar & onwind carriers, select the class number ('all' to plot all classes individually)
resource_class = 'all'

### Define period
start = '2023-03-01'
end = '2023-03-10'



#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)

if ('wind' in carrier or 'solar' in carrier) and resource_class != 'all':
    ggt_p.loc[start:end].filter(like=f'{resource_class} {carrier}').plot(ax=ax, alpha=.7, legend=False, linewidth=.5)
else:
    ggt_p.loc[start:end].filter(like=carrier).plot(ax=ax, alpha=.7, legend=False, linewidth=.5)

ax.grid(True, linestyle='--', alpha=0.25)
ax.set_ylabel('MW')

#### Summary by resource class

Solar and wind carries may have classes. Show a table with a feature per bus and class.

In [ ]:
#################### Parameters

### Select carrier
carrier = 'onwind'

### Select feature (uncomment one of the following):
feature = 'AEP'



#################### Table
if ('wind' in carrier or 'solar' in carrier):   
    ### Filter
    ggt_p_filtered = ggt_p.filter(like=carrier, axis=1).sum().to_frame(name='AEP')
    ### Add class and bus columns. Use multi-index with 'bus' and 'carrier'    
    split_index = ggt_p_filtered.index.to_series().str.rsplit(' ', n=2, expand=True)
    ggt_p_filtered['bus'] = split_index[0].values
    ggt_p_filtered['resource_class'] = split_index[1].astype(int) # resource_class as integer    
    ### MWh to TWh
    ggt_p_filtered[['AEP']] = ggt_p_filtered[['AEP']]/1000000
    ### Make pivot
    ggt_p_pivot = ggt_p_filtered.pivot(index="bus", columns="resource_class", values=feature)

    styled_table = ggt_p_pivot.style \
        .set_caption(f"Table with {feature} [TWh] for {carrier}, according to bus and class.") \
        .format("{:.3f}") \
        .background_gradient(cmap="Blues")
    
    display(styled_table)
else:
    print(f'Carrier {carrier} is not solar or wind type.. does it contain pmax_pu and classes?')


Add a bar plot showing `AEP`

In [ ]:
#################### Parameters

## bar width
width = 0.3 



#################### Figure

df_sorted = ggt_p_filtered.sort_values(["bus", "resource_class"])

buses = df_sorted["bus"].unique()
resources = df_sorted["resource_class"].unique()

x = np.arange(len(buses))                 
offsets = np.linspace(-width/2.5, width/2.5, len(resources))  


fig, ax = plt.subplots(figsize=(12, 8))

colors = {
    "AEP": "#93c543",
}


metrics_order = ["AEP"]

# To control added labels
labels_added = set()

for i, res in enumerate(resources):
    df_res = df_sorted[df_sorted["resource_class"] == res]
    pos = x + offsets[i]
    
    for metric in metrics_order:
        label = metric if metric not in labels_added else None
        ax.bar(
            pos,
            df_res[metric],
            width=width/len(resources),
            color=colors[metric],
            alpha=0.7,
            label=label
        )
        labels_added.add(metric)

ax.set_xticks(x)
ax.grid(axis="y", linestyle="--", alpha=0.6)
ax.set_xlabel("Bus", fontsize=16)
ax.set_ylabel("AEP [TWh]", fontsize=16)
ax.set_xticklabels(buses, fontsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.legend(fontsize=14)

plt.tight_layout()


#### Maps

Plot a map showing a particular feature of the potential generation for a carrier at each region.

In [ ]:
#################### Parameters

##### Select carrier:
carrier = 'onwind'

### For solar & onwind carriers, select the class number ('all' to aggregate all classes)
resource_class = 'all'

##### Select feature (uncomment one of the following):
feature = 'AEP' 



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Define gdf_regions
if 'off' in carrier:
    gdf_regions = gdf_regions_offshore
else:
    gdf_regions = gdf_regions_onshore


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_generatorst_p(carrier, n, feature, ax, gdf_regions, resource_class, params['map_network_generatorst_p'], params_local)

### Variable: `n.global_constraints`

Place `n.global_constraints` in a dataFrame and show its content.

In [ ]:
gc = n.global_constraints

gc

### Variable: `n.lines`

Place `n.lines` in a dataFrame and show its content.

In [ ]:
ln = n.lines

ln.head()

How is the distribution of line lengths?

In [ ]:
#################### Parameters
bins = 50



#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)


ax.hist(ln['length'], bins=bins, edgecolor='black')

ax.set_xlabel('km')
ax.grid(True, linestyle='--', alpha=0.5)

How is the relationship between line capital costs and line length?

In [ ]:
#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)


ln.plot(ax=ax, kind='scatter', x='length', y='capital_cost')

ax.set_xlabel('km')
ax.set_ylabel('EUR/MW')
ax.grid(True, linestyle='--', alpha=0.5)


ratio = ln['capital_cost']/ln['length']

print(f'The ratio values of capital cost vs. length  are {ratio.round(2).unique()} EUR/(MW·km)')

### Variable: `n.links`

Place `n.links` in a dataFrame and show its content.

In [ ]:
lk = n.links

lk.head()

Show the carriers associated with links.

In [ ]:
lk['carrier'].drop_duplicates().reset_index(drop=True)

#### Link type: DC

Place DC links in a dataFrame and show its content. Include interconnections.

In [ ]:
lk_DC = lk[lk['carrier'].isin(['DC', 'DC_ic export', 'DC_ic import'])]

lk_DC.head()

How is the distribution of DC link lengths?

In [ ]:
#################### Parameters
bins = 50



#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)


ax.hist(lk_DC['length'], bins=bins, edgecolor='black')

ax.set_xlabel('km')
ax.grid(True, linestyle='--', alpha=0.5)

How is the relationship between DC link capital costs and link length?

In [ ]:
#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)


lk_DC.plot(ax=ax, kind='scatter', x='length', y='capital_cost')

ax.set_xlabel('km')
ax.set_ylabel('EUR')
ax.grid(True, linestyle='--', alpha=0.5)


ratio = lk_DC['capital_cost']/lk_DC['length']

print(f'The ratio values of capital cost vs. length  are {ratio.round(2).unique()} EUR/km')

#### Link types: CCGT, OCGT, H2 Electrolysis, H2 Fuell cell, H2 turbine, battery charger, battery discharger

**NOTE**: Links represent different elements in the network. The aggregated capacity per carrier may need to be divided by the efficiency to properly reflect the capacity. For example, for CCGT, the link represents the conversion from gas to electricity, and the associated capacity refers to the input node. However, the CCGT capacity refers to the power capacity (output node).

In the following, **"_e"** is added to express "electric" (power output, the efficiency is included).

##### Maps

Plot a map showing a particular feature of a link carrier at each region.

Then, show another map with the same information aggregated to a certain NUTS level (weighted with ovelap area).

In [ ]:
#################### Parameters

### Select carrier (uncomment one of the following)
carrier = 'CCGT'
# carrier = 'OCGT'
# carrier = 'H2 Electrolysis'
# carrier = 'H2 Fuel Cell'
# carrier = 'battery charger'
# carrier = 'battery discharger'
# carrier = 'H2 turbine'


### Select feature (uncomment one of the following):
# feature = 'area' 
# feature = 'p_nom_e'
feature = 'p_nom_e_opt'



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_links(carrier, n, feature, ax, gdf_regions_onshore, params['map_network_links'], params_local)

In [ ]:
#################### Parameters

### Select NUTS level
NUTS_level = 3



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Define NUTS file
if NUTS_level==2:
    gdf_NUTS = gdf_NUTS2
if NUTS_level==3:
    gdf_NUTS = gdf_NUTS3    


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature aggregated at NUTS regions
xp.map_NUTS_links(carrier, n, f'{feature}_NUTS', ax, gdf_regions_onshore, gdf_NUTS, params['map_NUTS_links'], params_local)

### Variable: `n.loads`

Place `n.loads` in a dataFrame and show its content.

In [ ]:
lo = n.loads

lo.head()

#### Summary

What is the aggregated annual demand per carrier?

In [ ]:
lo['unit'] = lo['bus'].map(n.buses['unit'])

lo.groupby('carrier').agg(
    Total_demand=pd.NamedAgg(column='p_set', aggfunc='sum'),
    Unit=pd.NamedAgg(column='unit', aggfunc='first'),
    Buses=pd.NamedAgg(column='p_set', aggfunc='size'),
)


### Variable: `n.loads_t[p_set]`

Place `n.loads_t[p_set]` in a dataFrame and show its content.

In [ ]:
lot_pset = n.loads_t['p_set']

lot_pset.head()

#### Electricity

Filter `lot_pset` for electricity sector.

In [ ]:
electricity_key = 'electricity'
lot_pset_electricity = xp.filter_df_columns(lot_pset, filter_key=electricity_key)

lot_pset_electricity.head()

##### Time series

How do the load time series look like?

In [ ]:
#################### Parameters
start = '2013-01-01'
end = '2013-01-31'


#################### Figure
fig_size = [10,4]
fig, ax = plt.subplots(figsize=fig_size)

# Use full time series to compute totals
ts_full = lot_pset_electricity

# Plot only selected interval
ts = ts_full.loc[start:end]
ts.plot(ax=ax, alpha=.8, legend=False, linewidth=.5)

# Convert MW time series to TWh for legend and title
snapshot_hours = ts_full.index.to_series().diff().dt.total_seconds().div(3600).median()
if pd.isna(snapshot_hours):
    snapshot_hours = 1.0

totals_twh = ts_full.sum() * snapshot_hours / 1e6
total_demand_twh = totals_twh.sum()

legend_labels = [f"{col} ({totals_twh[col]:.2f} TWh)" for col in ts.columns]
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, legend_labels, loc='best', fontsize=8)

ax.set_title(f"Electricity demand time series (total: {total_demand_twh:.2f} TWh)")
ax.grid(True, linestyle='--', alpha=1)
ax.set_ylabel('MW')

##### Maps

Plot a map showing a particular feature of the load at each region.

In [ ]:
#################### Parameters

### Select feature (uncomment one of the following):
# feature = 'area' 
feature = 'annual_load'
#feature = 'annual_load_density'

### Select load key
electricity_key = 'electricity'


#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value


#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_loadst_pset(
    n,
    feature,
    ax,
    gdf_regions_onshore,
    params['map_network_loadst_pset'],
    params_local,
    load_key=electricity_key,
 )

#### Heating

Filter `lot_pset` for heating sector.

In [ ]:
heating_key = 'heating'
lot_pset_heating = xp.filter_df_columns(lot_pset, filter_key=heating_key)

lot_pset_heating.head()

##### Time series

Aggregate regions, and plot time series according to urban central, urban decentral and rural.

In [ ]:
#################### Parameters
start = '2013-01-01'
end = '2013-01-31'

heating_key = 'heating'
heating_patterns = xp.get_column_patterns(heating_key)

#################### Data aggregation by pattern
lot_pset_heating_agg = pd.DataFrame(index=lot_pset_heating.index)
missing_patterns = []

for pattern in heating_patterns:
    matched_cols = lot_pset_heating.columns[
        lot_pset_heating.columns.str.contains(pattern, case=False, na=False)
    ]
    if len(matched_cols) == 0:
        missing_patterns.append(pattern)
    else:
        lot_pset_heating_agg[pattern] = lot_pset_heating[matched_cols].sum(axis=1)

if missing_patterns:
    print(f'Patterns not found and skipped: {missing_patterns}')

if lot_pset_heating_agg.empty:
    print('No series could be built from the requested patterns.')
else:
    # Convert MW time series to TWh using snapshot duration in hours
    snapshot_hours = (
        lot_pset_heating_agg.index.to_series().diff().dt.total_seconds().div(3600).median()
    )
    if pd.isna(snapshot_hours):
        snapshot_hours = 1.0
    totals_twh = lot_pset_heating_agg.sum() * snapshot_hours / 1e6
    total_demand_twh = totals_twh.sum()

    legend_labels = [
        f"{col} ({totals_twh[col]:.2f} TWh)" for col in lot_pset_heating_agg.columns
    ]

    #################### Figure
    fig_size = [11, 5]
    fig, ax = plt.subplots(figsize=fig_size)

    lot_pset_heating_agg.loc[start:end].plot.area(
        ax=ax,
        stacked=True,
        alpha=0.9,
    )

    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, legend_labels, loc='best')

    ax.set_title(f'Heating demand time series (total: {total_demand_twh:.2f} TWh)')
    ax.set_ylabel('MW')
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()

##### Maps

Plot a map showing a particular feature of the heat load at each region.

In [ ]:
#################### Parameters

# Select feature (uncomment one of the following):
# feature = 'area'
# feature = 'annual_load'
feature = 'annual_load_density'

# Use key-based filtering + aggregation by node
heating_key = 'heating'

#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value
params_local['vmax'] = ''   # Leave empty for automatic value

#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})

### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add heating load feature at regions
xp.map_network_loadst_pset(
    n,
    feature,
    ax,
    gdf_regions_onshore,
    params['map_network_loadst_pset'],
    params_local,
    load_key=heating_key,
)

### Variable: `n.storage_units`

Place `n.storage_units` in a dataFrame and show its content.

In [ ]:
su = n.storage_units

su.head()

#### Summary

What is the aggregated capacity per carrier? 

How many buses have storage units for each carrier? How many of them have zero capacity?

In [ ]:
su.groupby('carrier').agg(
    Total_capacity=pd.NamedAgg(column='p_nom', aggfunc='sum'),
    Buses=pd.NamedAgg(column='p_nom', aggfunc='size'),
    Buses_zero_capacity=pd.NamedAgg(column='p_nom', aggfunc=lambda x: (x == 0).sum()),
    Buses_non_zero_capacity=pd.NamedAgg(column='p_nom', aggfunc=lambda x: (x != 0).sum()),
)

#### Maps

Plot a map showing a particular feature of a storage unit carrier at each region.

Then, show another map with the same information aggregated to a certain NUTS level (weighted with ovelap area).

In [ ]:
#################### Parameters

### Select carrier
carrier = 'PHS'

### Select feature (uncomment one of the following):
# feature = 'area' 
# feature = 'p_nom'
# feature = 'p_nom_density'
feature = 'max_hours'



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_storage_units(carrier, n, feature, ax, gdf_regions_onshore, params['map_network_storage_units'], params_local)

In [ ]:
#################### Parameters

### Select NUTS level
NUTS_level = 2



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Define NUTS file
if NUTS_level==2:
    gdf_NUTS = gdf_NUTS2
if NUTS_level==3:
    gdf_NUTS = gdf_NUTS3    


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature aggregated at NUTS regions
xp.map_NUTS_storage_units(carrier, n, f'{feature}_NUTS', ax, gdf_regions_onshore, gdf_NUTS, params['map_NUTS_storage_units'], params_local)

#### Maximum hours

What is the relation between installed capacity and max. hours, for each carrier?

In [ ]:
#################### Figure
fig_size = [8,4]
fig, ax = plt.subplots(figsize=fig_size)

tech_colors = n.carriers['color']

for carrier, group in su.groupby('carrier'):
    ax.scatter(group['p_nom'], group['max_hours'], label=carrier, color=tech_colors[carrier], alpha=0.7)

ax.set_xlabel('Installed capacity [MW]')
ax.set_ylabel('Max. hours')
ax.set_title('Storage units')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()

### Variable: `n.stores`

Place `n.stores` in a dataFrame and show its content.

In [ ]:
st = n.stores

st.head()

Show the carriers associated with stores.

In [ ]:
st['carrier'].drop_duplicates().reset_index(drop=True)

#### Maps

Plot a map showing a particular feature of a store carrier at each region.

Then, show another map with the same information aggregated to a certain NUTS level (weighted with ovelap area).

In [ ]:
#################### Parameters

### Select carrier (uncomment one of the following):
carrier = 'battery'
# carrier = 'H2 Store'

### Select feature (uncomment one of the following):
feature = 'e_nom_opt'



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature at regions
xp.map_network_stores(carrier, n, feature, ax, gdf_regions_onshore, params['map_network_stores'], params_local)

In [ ]:
#################### Parameters

### Select NUTS level
NUTS_level = 3



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value 
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Define NUTS file
if NUTS_level==2:
    gdf_NUTS = gdf_NUTS2
if NUTS_level==3:
    gdf_NUTS = gdf_NUTS3    



### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Add network feature aggregated at NUTS regions
xp.map_NUTS_stores(carrier, n, f'{feature}_NUTS', ax, gdf_regions_onshore, gdf_NUTS, params['map_NUTS_stores'], params_local)

### Analysis: Capacity expansion

Make a summary of initial and optimal capacities.

**Note:** Capacity associated to links `CCGT`, `OCGT`, `H2 Fuel Cell` and `battery discharger` have been multiplied by efficiencies to consider output power.

In [ ]:
#################### Get summary

df_capacities = xp.df_network_capacities(n)

df_capacities

Plot for a selection of carriers.

In [ ]:
#################### Parameters
carrier_list = ['solar', 'onwind', 'offwind-float', 'CCGT', 'DC', 'battery charger' ]


#################### Figure
fig_size = [10,6]

fig, ax = plt.subplots(figsize=fig_size)

df_capacities.loc[carrier_list].plot(kind='bar',rot=45, ax=ax)

ax.grid(True, linestyle='--', linewidth=0.5, color='black', alpha=0.25)

### Analysis: Grid expansion

If enabled, optimation includes grid expansion for lines and links.

Where was grid expansion required?

Plot the network showing the increase of line capacities (if any).

In [ ]:
#################### Parameters
line_widths = 1*(n.lines.s_nom_opt-n.lines.s_nom) / 1e3
link_widths = 1*(n.links.p_nom_opt-n.links.p_nom) / 1e3



#################### Figure
fig_size = [12,12]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Add network
n.plot(ax=ax, 
       line_widths=line_widths, 
       link_widths=link_widths, 
       bus_sizes=params['bus_sizes'], 
       bus_colors=params['bus_colors'], 
       boundaries=params[f'boundaries_offshore_{spatial_domain}'])

### Add regions_onshore
xp.map_add_region(ax, gdf_regions_onshore, params['map_add_region'])

### Add regions_offshore
xp.map_add_region(ax, gdf_regions_offshore, params['map_add_region'], is_offshore=True)

### Add map features
xp.map_add_features(ax, params['map_add_features'])

### Analysis: Grid congestion

Where are the bottlenecks in the grid from the optimal dispatch?

Plot the network showing the level of congestion.

In [ ]:
#################### Parameters

### Select criterion
# cong_criterion = ('mean', 'na')
cong_criterion = ('quantile', 0.9)


### Define line widths
line_widths = 1*(n.lines.s_nom_opt) / 1e3

##### WIP
### Define link widths
# first all at zero, then change those that are relevant
link_widths = 0*(n.links.p_nom_opt) / 1e3 
# # Set values to DC links and DC_ic_export links (export links will later include link activity in both directions)
# lk_DC = lk[lk['carrier'].isin(['DC', 'DC_ic export'])]
# link_widths.loc[lk_DC.index] = 1 * lk_DC['p_nom_opt'] / 1e3


# #################### Time series with line loads (positive and negative)
lnt_p0 = n.lines_t['p0']
##### WIP
# lkt_p0 = n.links_t['p0']
##### WIP



# ##### for interconnections
# # 
# for interconnection in n.links.loc[n.links.index.str.contains('export')].index:
#     try:
#         join_power = lkt_p0[interconnection] + lkt_p0[interconnection.replace('export', 'import')]
#         lkt_p0[interconnection] = join_power        
#     except KeyError:
#         join_power = 0
        
    


# # for link with Balearic islands
# for interconnection in n.links.loc[n.links.index.str.contains('relation')].index:
#     if 'reversed' not in interconnection:
#         try:
#             join_power = lkt_p0[interconnection] - lkt_p0[f'{interconnection}-reversed']
#             lkt_p0[interconnection] = join_power
#             lkt_p0[f'{interconnection}-reversed'] = join_power
#         except KeyError:
#             join_power = 0




#################### Get the line_colors

if cong_criterion[0] == 'mean':
    ln_result_criterion = lnt_p0.abs().mean() / n.lines['s_nom']
    # lk_result_criterion = lkt_p0.abs().mean() / n.links['p_nom']
    title_cb = 'Mean of line CF'
if cong_criterion[0] == 'quantile':
    ln_result_criterion = lnt_p0.abs().quantile(cong_criterion[1]) / n.lines['s_nom']
    # lk_result_criterion = lkt_p0.abs().quantile(cong_criterion[1]) / n.links['p_nom']
    title_cb = f'Q{cong_criterion[1]} of line CF'

ln_result_criterion

norm = mcolors.Normalize(vmin=0, vmax=0.7)
cmap = plt.cm.rainbow  # You can choose any colormap you like

ln_colors = [cmap(norm(p)) for p in ln_result_criterion]
# lk_colors = [cmap(norm(p)) for p in lk_result_criterion]



#################### Figure
fig_size = [12,12]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Add network
n.plot(ax=ax, 
       line_widths=line_widths, 
       link_widths=link_widths, 
       line_colors=ln_colors, 
       #link_colors=lk_colors,
       bus_sizes=params['bus_sizes'], 
       bus_colors=params['bus_colors'], 
       boundaries=params[f'boundaries_offshore_{spatial_domain}']
       )


# Create a ScalarMappable for the colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Required for compatibility with colorbar creation
# Add the colorbar to the figure
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', shrink=0.55, pad=0.02)
cbar.set_label(title_cb, fontsize=12)


### Add regions_onshore
xp.map_add_region(ax, gdf_regions_onshore, params['map_add_region'])

### Add map features
xp.map_add_features(ax, params['map_add_features'])




### Analysis: Interconnections

This analysis is only for PyPSA-Spain, with the interconnections enabled.

What are the energy imports and exports?

In [ ]:
exports_t = n.links_t['p0'].filter(like='export')
imports_t = -n.links_t['p1'].filter(like='import')

import_sums = imports_t.div(1e6).sum()
export_sums = exports_t.div(1e6).sum()

import_bar = pd.DataFrame(import_sums).T
export_bar = pd.DataFrame(export_sums).T

import_bar.index = ['Imports']
export_bar.index = ['Exports']

combined = pd.concat([import_bar, export_bar])


#################### Colors
ic_color_dict = xp.get_ic_colors(combined.columns)
colors = [ic_color_dict[col] for col in combined.columns]


#################### Figure
fig_size = [6, 6]
fig, ax = plt.subplots(figsize=fig_size)

combined.plot(
    kind='bar',
    stacked=True,
    color=colors,
    ax=ax,
    width=0.7
)

ax.set_ylabel("TWh")
ax.set_xlabel("")
ax.grid(True, linestyle='--', linewidth=0.5, color='black', alpha=0.25)
ax.legend(
    title='Border links',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
#################### Parameters
start = '2023-03-01'
end = '2023-03-15'



#################### Figure
# Exports positive, imports negative
exports_plot = exports_t.loc[start:end].div(1e3).abs()
imports_plot = -imports_t.loc[start:end].div(1e3).abs()

export_colors = [ic_color_dict.get(col, '#808080') for col in exports_plot.columns]
import_colors  = [ic_color_dict.get(col, '#808080') for col in imports_plot.columns]

fig_size = [10, 4]
fig, ax = plt.subplots(figsize=fig_size)

# Stacked areas above 0 (exports)
ax.stackplot(
    exports_plot.index,
    exports_plot.T.values,
    labels=[f"{col} (export)" for col in exports_plot.columns],
    colors=export_colors,
    alpha=0.85,
)

# Stacked areas below 0 (imports)
ax.stackplot(
    imports_plot.index,
    imports_plot.T.values,
    labels=[f"{col} (import)" for col in imports_plot.columns],
    colors=import_colors,
    alpha=0.45,
)

ax.axhline(0, color='black', linewidth=0.8, alpha=0.7)
ax.set_ylabel('Power [GW]')
ax.set_xlabel('')
ax.grid(True, linestyle='--', linewidth=0.5, color='black', alpha=0.25)
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=False,
    fontsize=9,
)
plt.tight_layout()

How do marginal prices vary throughout the year in electricity buses?

In [ ]:
#################### Parameters
start = '2023-03-01'
end = '2023-03-15'

color_ES = 'tab:red'
color_PT = mcolors.to_hex(plt.cm.Greens(0.7))
color_FR = mcolors.to_hex(plt.cm.Blues(0.7))

alpha_ES = 0.2
alpha_PT = 0.9
alpha_FR = 0.9



#################### Figure
df = n.buses_t.marginal_price.copy().loc[start:end]

cols_ES = [c for c in df.columns if c in gdf_regions_onshore["name"].values]
cols_PT = [c for c in df.columns if c=="PT"]
cols_FR = [c for c in df.columns if c=="FR"]

fig, ax = plt.subplots(figsize=(10, 4))

# ES electricity buses
for i, column in enumerate(cols_ES):
    ax.plot(
        df.index,
        df[column],
        color=color_ES,
        alpha=alpha_ES,
        linewidth=1,
        label='ES' if i == 0 else None,
    )

# PT
for i, column in enumerate(cols_PT):
    ax.plot(
        df.index,
        df[column],
        color=color_PT,
        alpha=alpha_PT,
        linewidth=3,
        label='PT' if i == 0 else None,
    )

# FR
for i, column in enumerate(cols_FR):
    ax.plot(
        df.index,
        df[column],
        color=color_FR,
        alpha=alpha_FR,
        linewidth=3,
        label='FR' if i == 0 else None,
    )

ax.set_title('')
ax.set_ylabel('Marginal price (EUR/MWh)')
ax.set_xlabel('')
ax.legend()
ax.grid(alpha=0.2)
plt.show()